# Garmin Connect API Exploration

Interactive testing ground for the `garminconnect` dependency ahead of building `src/ingestion/garmin_client.py`.

**Goals**

1. Confirm authentication works using credentials from `src/core/config.py` (backed by `.env`).
2. Focus area: downloading **FIT files** for activities and wellness/monitoring data (per the architecture doc's Bronze-layer "FIT vs JSON" investigation).
3. Survey the rest of the 150+ method API surface so we know what's available for later ingestion work.

**Before running:**

- Copy [`.env.template`](../.env.template) to `.env` at the repo root and fill in `GARMIN__EMAIL` / `GARMIN__PASSWORD`.
- Cells in this notebook make **live calls to your real Garmin Connect account** and will write real personal health/activity data to `data/_scratch_garmin_test/` (git-ignored). Nothing here is committed to version control.
- On first login you may be prompted for an MFA code inline (`input()`); afterwards, OAuth tokens are cached (`GARMIN__TOKENSTORE_PATH`, default `~/.garminconnect`) so subsequent runs skip credential login.


In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError("Could not locate repository root (pyproject.toml not found).")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(
        f"No .env file found at {env_path}. Copy .env.template to .env and fill in "
        "your Garmin credentials before running this notebook."
    )

from src.core.config import get_settings

settings = get_settings()

# Scratch area for this notebook's downloads only — never used by the real
# ingestion pipeline (which will write to settings.data.bronze_dir).
SCRATCH_DIR = REPO_ROOT / "data" / "_scratch_garmin_test"
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root:     {REPO_ROOT}")
print(f"Garmin account: {settings.garmin.email is not None and 'CONFIGURED' or 'MISSING'}")
print(f"Token cache:    {settings.garmin.tokenstore_path}")
print(f"Scratch dir:    {SCRATCH_DIR}")


Repo root:     c:\Users\gabri\workspace\strider
Garmin account: CONFIGURED
Token cache:    C:\Users\gabri\.garminconnect
Scratch dir:    c:\Users\gabri\workspace\strider\data\_scratch_garmin_test


## Authentication

`Garmin.login()` uses the same mobile SSO flow as the Android app. `prompt_mfa` is only invoked if
your account has MFA enabled. Passing a `tokenstore` path lets the library cache tokens on disk and
auto-refresh them, so re-running this notebook later won't require a fresh login.


**⚠️ Garmin's SSO login endpoint is IP-rate-limited.** A fresh credential login (no cached tokens

yet) can trigger a `429 Too Many Requests` on the very first attempt, even before you've donetypically IP-based and needs time to clear regardless of what this code does.

anything wrong — the library tries several login strategies in a row, which compounds the risk.If you hit the cooldown, **wait** rather than passing `force=True` immediately — Garmin's block is

Re-running a failed login immediately makes this worse, not better.

- Once tokens are cached successfully, subsequent runs load them from disk with no fresh SSO call.

To avoid this, authentication goes through `src/ingestion/garmin_client.py`, which:  `rate_limit.login_cooldown_seconds` in `src/core/config.py` (default 1 hour).

- Skips `login()` entirely if this session is already authenticated (safe to re-run this cell).  attempts (even across kernel restarts) until the cooldown elapses — see
- Persists a **cooldown marker** next to the token cache after any 429, and refuses further login

In [2]:
import garminconnect as gc
from garminconnect import Garmin, GarminConnectAuthenticationError, GarminConnectTooManyRequestsError

from src.ingestion.garmin_client import GarminAuthCooldownError, build_client, login

client = build_client(settings)

try:
    login(client, settings)
    print(
        "Authenticated"
        + (" (session already active)." if client.client.is_authenticated else ".")
    )
except GarminAuthCooldownError as e:
    print(f"Login blocked by cooldown from a prior rate-limit: {e}")
    raise
except GarminConnectAuthenticationError as e:
    print(f"Authentication failed — check GARMIN__EMAIL / GARMIN__PASSWORD in .env: {e}")
    raise
except GarminConnectTooManyRequestsError as e:
    print(f"Rate limited by Garmin Connect — cooldown recorded, wait before retrying: {e}")
    raise


Authenticated (session already active).


## Profile & daily stats (JSON)

Quick sanity checks that the session works, using cheap, low-volume endpoints.


In [3]:
from datetime import date
from datetime import timedelta

yesterday = (date.today() - timedelta(days=1)).isoformat()

print("Unit system:", client.get_unit_system())

stats = client.get_stats(yesterday)
print(f"\nYesterday ({yesterday}):")
print(f"  Steps:        {stats.get('totalSteps')}")
print(f"  Calories:     {stats.get('totalKilocalories')}")
print(f"  Resting HR:   {stats.get('restingHeartRate')}")
print(f"  Stress avg:   {stats.get('averageStressLevel')}")

Unit system: metric

Yesterday (2026-09-25):
  Steps:        9790
  Calories:     2300.0
  Resting HR:   60
  Stress avg:   33


Test downloading a simple activity and exploring the downloaded data

In [4]:
last_activity = client.get_activities(start=0,limit=2, activitytype="running")

In [5]:
last_activity[0]

{'activityId': 24480686682,
 'activityName': 'Lodi - Speed Repeats',
 'description': 'male piede',
 'startTimeLocal': '2026-09-24 12:53:35',
 'startTimeGMT': '2026-09-24 10:53:35',
 'activityType': {'typeId': 1,
  'typeKey': 'running',
  'parentTypeId': 17,
  'isHidden': False,
  'restricted': False,
  'trimmable': True},
 'eventType': {'typeId': 9, 'typeKey': 'uncategorized', 'sortOrder': 10},
 'isFavorite': False,
 'isPR': False,
 'isAutoCalcCalories': False,
 'isParent': False,
 'distance': 6011.75,
 'duration': 2210.212890625,
 'elapsedDuration': 2336.660888671875,
 'movingDuration': 2203.2130184173584,
 'elevationGain': 27.724830627441406,
 'elevationLoss': 23.480682373046875,
 'averageSpeed': 2.720000028610229,
 'maxSpeed': 4.2270002365112305,
 'startLatitude': 45.31171535141766,
 'startLongitude': 9.508008258417249,
 'hasPolyline': True,
 'hasImages': False,
 'ownerId': 89118819,
 'ownerDisplayName': '2773b3cd-4bf5-4e51-bca4-b7ac1e7d1046',
 'ownerFullName': 'Gabriele Di Corato',

## Activities & FIT downloads (primary focus)

`download_activity(activity_id, dl_fmt=Garmin.ActivityDownloadFormat.ORIGINAL)` returns the
**original binary FIT file, zipped**, exactly as recorded by the device — this is what
`src/processing/parser_fit.py` (via `fitdecode`) should consume for the Silver layer, per the
architecture doc's directive to prefer FIT over JSON where available. Other formats (`TCX`, `GPX`,
`KML`, `CSV`) are lossy derivatives and not used by the pipeline.


In [ ]:
total = client.count_activities()
print(f"Total activities on account: {total}\n")

recent = client.get_activities(start=0, limit=5)
for act in recent:
    activity_type = act.get("activityType", {}).get("typeKey", "?")
    print(
        f"{act['activityId']:>12}  {act.get('startTimeLocal', '?'):>20}  "
        f"{act.get('activityName', '?'):<30}  {activity_type}"
    )


Total activities on account: 1246

 24507581501   2026-09-26 17:17:46  Lodi Walking                    walking
 24501751461   2026-09-26 10:06:06  Lodi Walking                    walking
 24501749482   2026-09-26 00:20:07  Lodi Walking                    walking
 24484858072   2026-09-24 18:52:10  Lodi Walking                    walking
 24480686682   2026-09-24 12:53:35  Lodi - Speed Repeats            running


In [32]:
import zipfile

if not recent:
    print("No activities on the account to download.")
else:
    activity_id = recent[0]["activityId"]
    activity_name = recent[0].get("activityName", "unnamed")

    fit_zip_bytes = client.download_activity(
        activity_id, dl_fmt=Garmin.ActivityDownloadFormat.ORIGINAL
    )
    zip_path = SCRATCH_DIR / f"activity_{activity_id}.zip"
    zip_path.write_bytes(fit_zip_bytes)
    print(f"'{activity_name}' -> {len(fit_zip_bytes):,} bytes saved to {zip_path}")

    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            size = zf.getinfo(name).file_size
            print(f"  contains: {name} ({size:,} bytes)")


'Lodi Walking' -> 25,698 bytes saved to c:\Users\gabri\workspace\strider\data\_scratch_garmin_test\activity_24507581501.zip
  contains: 24507581501_ACTIVITY.fit (60,637 bytes)


## Wellness / monitoring data

Most day-level wellness metrics (`get_heart_rates`, `get_sleep_data`, `get_stress_data`, `get_hrv_data`,
`get_body_battery`, `get_spo2_data`, `get_respiration_data`, `get_training_status`, ...) are only
exposed as **JSON**, not FIT — per the architecture doc, these should land in Bronze as raw JSON and
then be shredded with Polars into Silver Parquet, rather than parsed as binary.

`download_health_snapshot(date)` is the one exception worth checking: despite its name, the endpoint
it calls is `/download-service/files/wellness/{date}` — the same family as the activity FIT download
route (`/download-service/files/activity/{id}`). We download it below to see whether it actually
contains a **binary monitoring FIT file** for the day (steps/HR/stress epochs), which would let us
prefer binary parsing for wellness data too, as the architecture doc directs us to investigate.


In [34]:
wellness_calls = {
    "get_heart_rates": client.get_heart_rates,
    "get_sleep_data": client.get_sleep_data,
    "get_stress_data": client.get_stress_data,
    "get_hrv_data": client.get_hrv_data,
    "get_body_battery": client.get_body_battery,
    "get_spo2_data": client.get_spo2_data,
    "get_respiration_data": client.get_respiration_data,
    "get_training_status": client.get_training_status,
    "get_training_readiness": client.get_training_readiness,
}

for name, fn in wellness_calls.items():
    try:
        result = fn(yesterday)
        if isinstance(result, dict):
            shape = f"dict with keys: {list(result.keys())}"
        elif isinstance(result, list):
            shape = f"list: {result}"
        else:
            shape = type(result).__name__
        print(f"{name:<24} -> {shape}")
    except Exception as e:
        print(f"{name:<24} -> ERROR: {e}")


get_heart_rates          -> dict with keys: ['userProfilePK', 'calendarDate', 'startTimestampGMT', 'endTimestampGMT', 'startTimestampLocal', 'endTimestampLocal', 'maxHeartRate', 'minHeartRate', 'restingHeartRate', 'lastSevenDaysAvgRestingHeartRate', 'heartRateValueDescriptors', 'heartRateValues']
get_sleep_data           -> dict with keys: ['dailySleepDTO', 'sleepMovement', 'remSleepData', 'sleepLevels', 'wellnessSpO2SleepSummaryDTO', 'wellnessEpochSPO2DataDTOList', 'wellnessEpochRespirationDataDTOList', 'wellnessEpochRespirationAveragesList', 'respirationVersion', 'sleepHeartRate', 'sleepStress', 'sleepBodyBattery', 'skinTempDataExists', 'bodyBatteryChange', 'restingHeartRate']
get_stress_data          -> dict with keys: ['userProfilePK', 'calendarDate', 'startTimestampGMT', 'endTimestampGMT', 'startTimestampLocal', 'endTimestampLocal', 'maxStressLevel', 'avgStressLevel', 'stressChartValueOffset', 'stressChartYAxisOrigin', 'stressValueDescriptorsDTOList', 'stressValuesArray', 'bodyBat

In [35]:
from datetime import timedelta

from garminconnect import GarminConnectNotFoundError, GarminConnectConnectionError

# Today's snapshot may not be finalized yet server-side; fall back to yesterday.
candidate_dates = [yesterday, (date.today() - timedelta(days=2)).isoformat()]

snapshot_bytes = None
snapshot_date = None
for d in candidate_dates:
    try:
        snapshot_bytes = client.download_health_snapshot(d)
        snapshot_date = d
        break
    except (GarminConnectNotFoundError, GarminConnectConnectionError) as e:
        print(f"No snapshot for {d}: {e}")

if snapshot_bytes is None:
    print("No health/wellness snapshot available for the recent dates tried.")
else:
    snapshot_path = SCRATCH_DIR / f"wellness_{snapshot_date}.zip"
    snapshot_path.write_bytes(snapshot_bytes)
    print(f"Saved {len(snapshot_bytes):,} bytes -> {snapshot_path}")

    with zipfile.ZipFile(snapshot_path) as zf:
        names = zf.namelist()
        print("Contents:", names)
        for name in names:
            marker = " <-- binary FIT!" if name.lower().endswith(".fit") else ""
            print(f"  - {name} ({zf.getinfo(name).file_size:,} bytes){marker}")


Saved 60,913 bytes -> c:\Users\gabri\workspace\strider\data\_scratch_garmin_test\wellness_2026-09-25.zip
Contents: ['486683316866_WELLNESS.fit', '486463062989_SLEEP_DATA.fit', '486322319116_METRICS.fit', '486169447434_SLEEP_DATA.fit', '486341705158_WELLNESS.fit', '486341709048_SLEEP_DATA.fit', '486481541994_WELLNESS.fit', '486683371025_SLEEP_DATA.fit', '486169384690_WELLNESS.fit', '486322315671_WELLNESS.fit', '486251181021_SLEEP_DATA.fit', '486405368524_SLEEP_DATA.fit', '486322368088_SLEEP_DATA.fit', '486463048031_WELLNESS.fit', '486481545189_SLEEP_DATA.fit', '486405350121_WELLNESS.fit', '486250975965_METRICS.fit', '486250961150_WELLNESS.fit', '486169389978_METRICS.fit']
  - 486683316866_WELLNESS.fit (3,551 bytes) <-- binary FIT!
  - 486463062989_SLEEP_DATA.fit (4,549 bytes) <-- binary FIT!
  - 486322319116_METRICS.fit (380 bytes) <-- binary FIT!
  - 486169447434_SLEEP_DATA.fit (3,337 bytes) <-- binary FIT!
  - 486341705158_WELLNESS.fit (3,267 bytes) <-- binary FIT!
  - 486341709048_SL

## Broader API surface

`garminconnect` exposes 150+ methods on the `Garmin` class. Rather than hardcoding a list that will
drift as the dependency is upgraded, introspect the installed version directly and bucket the public
methods by theme so we know what's available for future ingestion work beyond activities/wellness
(goals, gear, training plans, golf, nutrition, menstrual health, etc.).


In [36]:
import inspect

public_methods = sorted(
    name
    for name, member in inspect.getmembers(Garmin, predicate=inspect.isfunction)
    if not name.startswith("_")
)
print(f"Total public methods on Garmin: {len(public_methods)}")

keyword_categories = {
    "activities": ("activit",),
    "wellness/health": (
        "sleep", "stress", "hrv", "spo2", "respiration", "battery",
        "heart_rate", "heartrate", "hydration", "blood_pressure", "rhr",
    ),
    "training/fitness metrics": (
        "training", "endurance", "race", "fitnessage", "hill_score",
        "max_metrics", "lactate", "ftp", "cycling_ftp", "zones",
    ),
    "body composition/weight": ("weigh", "body_composition", "body_battery"),
    "gear/device": ("gear", "device"),
    "workouts/training plans": ("workout", "training_plan"),
    "goals/badges/challenges": ("goal", "badge", "challenge", "record"),
    "golf": ("golf",),
    "menstrual/pregnancy": ("menstrual", "pregnancy"),
    "nutrition": ("nutrition",),
    "account/system": ("login", "logout", "profile", "graphql", "download", "connectapi", "typed", "reload"),
}

categorized: set[str] = set()
categories: dict[str, list[str]] = {}
for label, keywords in keyword_categories.items():
    matches = [m for m in public_methods if any(k in m for k in keywords)]
    categories[label] = matches
    categorized.update(matches)
categories["other"] = [m for m in public_methods if m not in categorized]

for label, methods in categories.items():
    print(f"\n{label} ({len(methods)})")
    for m in methods:
        print(f"  - {m}")


Total public methods on Garmin: 159

activities (29)
  - add_gear_to_activity
  - count_activities
  - create_manual_activity
  - create_manual_activity_from_json
  - delete_activity
  - download_activity
  - get_activities
  - get_activities_by_date
  - get_activities_fordate
  - get_activity
  - get_activity_details
  - get_activity_exercise_sets
  - get_activity_gear
  - get_activity_hr_in_timezones
  - get_activity_power_in_timezones
  - get_activity_split_summaries
  - get_activity_splits
  - get_activity_typed_splits
  - get_activity_types
  - get_activity_weather
  - get_gear_activities
  - get_last_activity
  - import_activity
  - remove_gear_from_activity
  - set_activity_description
  - set_activity_exercise_sets
  - set_activity_name
  - set_activity_type
  - upload_activity

wellness/health (20)
  - add_hydration_data
  - delete_blood_pressure
  - get_all_day_stress
  - get_blood_pressure
  - get_body_battery
  - get_body_battery_events
  - get_heart_rate_zones
  - get_hear

## Findings & next steps

- **Activities → FIT**: `download_activity(id, dl_fmt=Garmin.ActivityDownloadFormat.ORIGINAL)` gives the
  original device-recorded FIT file (zipped). This is the Bronze payload for activities; `parser_fit.py`
  will `fitdecode` it directly into Silver records.
- **Wellness/monitoring → mostly JSON**: `get_heart_rates`, `get_sleep_data`, `get_stress_data`,
  `get_hrv_data`, `get_body_battery`, `get_spo2_data`, `get_respiration_data`, `get_training_status`, etc.
  only return JSON — check the cell output above for whether `download_health_snapshot` also yielded a
  binary `.fit` file. If it did, prefer that route for wellness Bronze storage per the architecture
  doc; if not (e.g. it's an ECG-only "Health Snapshot" rather than general monitoring data), fall back
  to the documented pattern: save raw JSON to Bronze, then shred with Polars into Silver.
- **Everything else** (goals/badges, gear, training plans, golf, nutrition, menstrual health) is
  available but out of scope for the current roadmap step — revisit once Bronze→Silver ingestion for
  activities and wellness is idempotent and operational.

Next: build `src/ingestion/garmin_client.py` (thin adapter wrapping the calls exercised above) and
`src/ingestion/ledger.py` (hash/timestamp tracking per `activity_id` / date so re-syncs are safe).


## Spike: Feature 2 — is plan adherence matching already solved by Garmin?

`ai_features.md` Feature 2 has an open question: does `garminconnect` expose Garmin Coach's assigned
workouts, and does a completed activity already carry a link back to the workout it was scheduled
from? If activities are already tagged with a `workoutId`, most of the "candidate matching" problem
in the design doc disappears — it becomes a lookup, not a reconciliation.

We check three things:
1. `get_scheduled_workouts` / `get_next_scheduled_workout` — can we read the assigned calendar at all?
2. The raw JSON of a recent activity — does it carry a workout/plan linkage field?
3. `get_training_plans` / `get_adaptive_training_plan_by_id` — is there a Garmin Coach adaptive plan
   object we could ingest as the "planned session" source?



In [ ]:
import json

# 1. Assigned calendar
try:
    scheduled = client.get_scheduled_workouts(year = 2026, month = 9)
    print(f"get_scheduled_workouts -> {len(scheduled) if isinstance(scheduled, list) else type(scheduled)} {len(scheduled)} items")
    if scheduled:
        print(f"keys: {scheduled.keys()}")
        print(json.dumps(scheduled, indent=2)[:1500])
except Exception as e:
    print(f"get_scheduled_workouts -> ERROR: {e}")

try:
    next_workout = client.get_next_scheduled_workout()
    print(f"\nget_next_scheduled_workout -> {next_workout}")
except Exception as e:
    print(f"\nget_next_scheduled_workout -> ERROR: {e}")

# 2. Does a completed activity carry a link back to a planned workout?
print("\n--- recent activity raw keys (looking for workout/plan linkage) ---")
candidate_keys = [k for k in recent[0].keys() if "workout" in k.lower() or "plan" in k.lower() or "coach" in k.lower()]
print(f"Candidate linkage keys on activity: {candidate_keys}")
print(json.dumps({k: recent[0][k] for k in candidate_keys}, indent=2, default=str))

# 3. Training plan objects
try:
    plans = client.get_training_plans()
    print(f"\nget_training_plans -> {len(plans) if isinstance(plans, list) else type(plans)}")
    if plans:
        print(json.dumps(plans[0], indent=2)[:1000])
except Exception as e:
    print(f"\nget_training_plans -> ERROR: {e}")


get_scheduled_workouts -> <class 'dict'> 6 items
{
  "startDayOfMonth": 2,
  "numOfDaysInMonth": 30,
  "numOfDaysInPrevMonth": 31,
  "month": 8,
  "year": 2026,
  "calendarItems": [
    {
      "id": 1789999992434,
      "groupId": null,
      "trainingPlanId": null,
      "itemType": "weight",
      "activityTypeId": null,
      "wellnessActivityUuid": null,
      "title": null,
      "date": "2026-09-21",
      "duration": null,
      "distance": null,
      "calories": null,
      "floorsClimbed": null,
      "avgRespirationRate": null,
      "unitOfPoolLength": null,
      "weight": 64599.0,
      "difference": null,
      "courseId": null,
      "courseName": null,
      "sportTypeKey": null,
      "url": null,
      "isStart": null,
      "isRace": null,
      "recurrenceId": null,
      "isParent": null,
      "parentId": null,
      "userBadgeId": null,
      "badgeCategoryTypeId": null,
      "badgeCategoryTypeDesc": null,
      "badgeAwardedDate": null,
      "badgeViewed": n

In [38]:
from datetime import date as _date

today_d = _date.today()

# get_scheduled_workouts needs year/month explicitly
try:
    scheduled_month = client.get_scheduled_workouts(today_d.year, today_d.month)
    print(f"get_scheduled_workouts({today_d.year}, {today_d.month}) -> {type(scheduled_month)}")
    print(json.dumps(scheduled_month, indent=2, default=str)[:1500])
except Exception as e:
    print(f"get_scheduled_workouts -> ERROR: {e}")

# Does the FULL single-activity payload (not the list summary) carry a workout linkage?
print("\n--- full get_activity() payload: workout/plan-related keys ---")
full_activity = client.get_activity(recent[0]["activityId"])
linkage_keys = [k for k in full_activity.keys() if "workout" in k.lower() or "plan" in k.lower() or "coach" in k.lower()]
print(f"Candidate keys: {linkage_keys}")
print(json.dumps({k: full_activity[k] for k in linkage_keys}, indent=2, default=str))

# Structured workout schema, using the workoutId surfaced by get_next_scheduled_workout above
try:
    workout_detail = client.get_workout_by_id(next_workout["workoutId"])
    print("\n--- get_workout_by_id (structured workout schema) ---")
    print(json.dumps(workout_detail, indent=2, default=str)[:2500])
except Exception as e:
    print(f"\nget_workout_by_id -> ERROR: {e}")


get_scheduled_workouts(2026, 9) -> <class 'dict'>
{
  "startDayOfMonth": 2,
  "numOfDaysInMonth": 30,
  "numOfDaysInPrevMonth": 31,
  "month": 8,
  "year": 2026,
  "calendarItems": [
    {
      "id": 1789999992434,
      "groupId": null,
      "trainingPlanId": null,
      "itemType": "weight",
      "activityTypeId": null,
      "wellnessActivityUuid": null,
      "title": null,
      "date": "2026-09-21",
      "duration": null,
      "distance": null,
      "calories": null,
      "floorsClimbed": null,
      "avgRespirationRate": null,
      "unitOfPoolLength": null,
      "weight": 64599.0,
      "difference": null,
      "courseId": null,
      "courseName": null,
      "sportTypeKey": null,
      "url": null,
      "isStart": null,
      "isRace": null,
      "recurrenceId": null,
      "isParent": null,
      "parentId": null,
      "userBadgeId": null,
      "badgeCategoryTypeId": null,
      "badgeCategoryTypeDesc": null,
      "badgeAwardedDate": null,
      "badgeViewed": 

In [39]:
# Narrow to workout-type calendar items, and check past (already-happened) ones for any
# completion/activity linkage field that isn't visible in the summary activity payload.
workout_items = [i for i in scheduled_month["calendarItems"] if i.get("itemType") == "workout"]
print(f"Workout calendar items this month: {len(workout_items)}")

past_items = [i for i in workout_items if i["date"] < today_d.isoformat()]
print(f"Of which in the past: {len(past_items)}\n")

for item in past_items:
    non_null = {k: v for k, v in item.items() if v not in (None, False)}
    print(json.dumps(non_null, indent=2, default=str))
    print("---")


Workout calendar items this month: 13
Of which in the past: 10

{
  "id": 1770510420,
  "itemType": "workout",
  "title": "Benchmark Run",
  "date": "2026-09-07",
  "sportTypeKey": "running",
  "atpPlanId": 1788790634,
  "workoutId": 1690884303,
  "protectedWorkoutSchedule": true
}
---
{
  "id": 1770657945,
  "itemType": "workout",
  "title": "Easy Run",
  "date": "2026-09-09",
  "sportTypeKey": "running",
  "atpPlanId": 1788790634,
  "workoutId": 1691034470
}
---
{
  "id": 1770657947,
  "itemType": "workout",
  "title": "Easy Run",
  "date": "2026-09-11",
  "sportTypeKey": "running",
  "atpPlanId": 1788790634,
  "workoutId": 1691034471
}
---
{
  "id": 1770657948,
  "itemType": "workout",
  "title": "Easy Run",
  "date": "2026-09-13",
  "sportTypeKey": "running",
  "atpPlanId": 1788790634,
  "workoutId": 1691034475
}
---
{
  "id": 1770657949,
  "itemType": "workout",
  "title": "Easy Run",
  "date": "2026-09-14",
  "sportTypeKey": "running",
  "atpPlanId": 1788790634,
  "workoutId": 16

In [40]:
# Since the calendar item carries no completion flag, cross-reference by date + type ourselves,
# exactly per the rule-based MVP proposed in ai_features.md Feature 2.
month_activities = client.get_activities_by_date(
    past_items[0]["date"], past_items[-1]["date"], activitytype="running"
)
acts_by_date = {a["startTimeLocal"][:10]: a for a in month_activities}

print(f"{'planned date':<12} {'title':<18} {'matched activity?':<20} name / duration")
for item in past_items:
    match = acts_by_date.get(item["date"])
    if match:
        mins = round(match["duration"] / 60, 1)
        print(f"{item['date']:<12} {item['title']:<18} YES                  '{match['activityName']}' ({mins} min)")
    else:
        print(f"{item['date']:<12} {item['title']:<18} NO MATCH")


planned date title              matched activity?    name / duration
2026-09-07   Benchmark Run      YES                  'Lodi - Benchmark Run' (9.0 min)
2026-09-09   Easy Run           YES                  'Lodi - Easy Run' (30.0 min)
2026-09-11   Easy Run           YES                  'Lodi - Easy Run' (30.0 min)
2026-09-13   Easy Run           NO MATCH
2026-09-14   Easy Run           YES                  'Lodi - Easy Run' (30.0 min)
2026-09-16   Easy Run           YES                  'Lodi - Easy Run' (30.1 min)
2026-09-18   Stride Repeats     YES                  'Lodi - Stride Repeats' (36.5 min)
2026-09-20   Long Easy Run      YES                  'Varazze - Long Easy Run' (70.0 min)
2026-09-21   Easy Run           YES                  'Lodi - Easy Run' (38.5 min)
2026-09-24   Speed Repeats      YES                  'Lodi - Speed Repeats' (36.8 min)


## Spike: new Gold-layer / AI-action signals worth surfacing

Read-only survey of the "training/fitness metrics" bucket that Feature 1's action catalog hasn't
considered yet — training readiness, race predictions, endurance/hill scores, lactate threshold,
running tolerance. These are candidate **inputs to new AI Actions**, not actions themselves.



In [41]:
signal_calls = {
    "get_training_readiness": lambda: client.get_training_readiness(today_d.isoformat()),
    "get_morning_training_readiness": lambda: client.get_morning_training_readiness(),
    "get_race_predictions": lambda: client.get_race_predictions(),
    "get_endurance_score": lambda: client.get_endurance_score(today_d.isoformat()),
    "get_hill_score": lambda: client.get_hill_score(today_d.isoformat()),
    "get_lactate_threshold": lambda: client.get_lactate_threshold(today_d.isoformat()),
    "get_running_tolerance": lambda: client.get_running_tolerance(today_d.isoformat()),
    "get_max_metrics": lambda: client.get_max_metrics(today_d.isoformat()),
    "get_nutrition_daily_food_log": lambda: client.get_nutrition_daily_food_log(today_d.isoformat()),
    "get_nutrition_daily_meals": lambda: client.get_nutrition_daily_meals(today_d.isoformat()),
    "get_nutrition_daily_settings": lambda: client.get_nutrition_daily_settings(),
}

for name, fn in signal_calls.items():
    try:
        result = fn()
        if isinstance(result, dict):
            shape = f"dict, keys: {list(result.keys())[:12]}"
        elif isinstance(result, list):
            shape = f"list[{len(result)}]" + (f", sample keys: {list(result[0].keys())[:12]}" if result and isinstance(result[0], dict) else "")
        else:
            shape = f"{type(result).__name__}: {result}"
        print(f"{name:<30} -> {shape}")
    except Exception as e:
        print(f"{name:<30} -> ERROR: {e}")


get_training_readiness         -> list[0]
get_morning_training_readiness -> ERROR: Garmin.get_morning_training_readiness() missing 1 required positional argument: 'cdate'
get_race_predictions           -> dict, keys: ['userId', 'fromCalendarDate', 'toCalendarDate', 'calendarDate', 'time5K', 'time10K', 'timeHalfMarathon', 'timeMarathon']
get_endurance_score            -> dict, keys: []
get_hill_score                 -> dict, keys: []
get_lactate_threshold          -> ERROR: Garmin.get_lactate_threshold() takes 1 positional argument but 2 were given
get_running_tolerance          -> ERROR: Garmin.get_running_tolerance() missing 1 required positional argument: 'enddate'
get_max_metrics                -> list[0]
get_nutrition_daily_food_log   -> dict, keys: ['mealDate', 'dayStartTime', 'dayEndTime', 'dailyViewType', 'mealDetails', 'loggedFoodsWithServingSizes']
get_nutrition_daily_meals      -> dict, keys: ['meals', 'dailyViewType']
get_nutrition_daily_settings   -> ERROR: Garmin.get_nutri

In [42]:
# Fix up call signatures for the ones that errored, and print real values for the
# most promising candidates (race predictions, nutrition).
print("race predictions:", json.dumps(client.get_race_predictions(), indent=2))

try:
    print("\nlactate threshold:", client.get_lactate_threshold())
except Exception as e:
    print(f"\nlactate threshold -> ERROR: {e}")

try:
    tolerance = client.get_running_tolerance(enddate=today_d.isoformat())
    print("\nrunning tolerance:", json.dumps(tolerance, indent=2, default=str)[:800])
except Exception as e:
    print(f"\nrunning tolerance -> ERROR: {e}")

try:
    morning = client.get_morning_training_readiness(today_d.isoformat())
    print("\nmorning training readiness:", json.dumps(morning, indent=2, default=str)[:800])
except Exception as e:
    print(f"\nmorning training readiness -> ERROR: {e}")

food_log = client.get_nutrition_daily_food_log(today_d.isoformat())
print("\nnutrition daily food log sample:", json.dumps(food_log, indent=2, default=str)[:600])


race predictions: {
  "userId": 89118819,
  "fromCalendarDate": null,
  "toCalendarDate": null,
  "calendarDate": "2026-09-26",
  "time5K": 1492,
  "time10K": 3282,
  "timeHalfMarathon": 7898,
  "timeMarathon": 18358
}

lactate threshold: {'speed_and_heart_rate': {'userProfilePK': None, 'version': None, 'calendarDate': None, 'sequence': None, 'speed': None, 'heartRate': None, 'heartRateCycling': None}, 'power': {}}

running tolerance -> ERROR: Garmin.get_running_tolerance() missing 1 required positional argument: 'startdate'

morning training readiness: null

nutrition daily food log sample: {
  "mealDate": "2026-09-26",
  "dayStartTime": "07:00:00",
  "dayEndTime": "22:00:00",
  "dailyViewType": "TIMELINE_VIEW",
  "mealDetails": [],
  "loggedFoodsWithServingSizes": []
}


## Spike findings — feeding back into `ai_features.md`

**Feature 2 (plan adherence) — the big open question is now answered:**
- `get_next_scheduled_workout()` / `get_scheduled_workouts(year, month)` **do** expose Garmin
  Coach's assigned calendar (`workoutId`, `atpPlanId`, `title`, `date`, `sportTypeKey`) — the plan
  side is readable.
- Calendar items carry **no completion/back-reference field**, even for past dates — Garmin does not
  expose a "was this done" flag or an `activityId` link. Reconciliation genuinely has to happen on
  our side, confirming the doc's framing.
- However, activities started *from* a scheduled workout on the device are auto-named
  `"{location} - {workout title}"` (confirmed: `"Lodi - Easy Run"`, `"Lodi - Speed Repeats"`, ...).
  Matching on **(date window + activity type + workout-title substring in activity name)** gave
  9/10 correct matches with zero ambiguity in this sample, and the 1 non-match was a real skipped
  session — a much stronger signal than the date+type+duration-only heuristic originally scoped.
  This should demote the "LLM-assisted ambiguous case" path to a true fallback for freeform (not
  device-started) activities, not the common case.
- `get_workout_by_id` confirms the structured schema needed for Feature 3's push side:
  `workoutSegments[].workoutSteps[]`, each with `stepType` (warmup/interval/recovery/...),
  `endCondition` (time/distance/lap-button), `endConditionValue`, and `targetType`
  (pace/HR-zone/power/no-target) — this is the shape Strider's plan generator must emit.
- `get_training_plans()` errored on this account; the working lead for "read the whole plan" is
  `atpPlanId` (seen on every calendar item) via `get_adaptive_training_plan_by_id`, not the generic
  training-plans endpoint — worth trying next before Feature 3 implementation.

**New signal candidates for Feature 1's action catalog:**
- `get_race_predictions()` returns real, already-computed 5K/10K/HM/Marathon time estimates — no
  extra Gold-layer math needed, just narration + trend-over-time storage. Good MVP action candidate
  ("Race Readiness").
- `get_training_readiness`, `get_endurance_score`, `get_hill_score`, `get_lactate_threshold` all
  returned empty/null for this account (likely needs more history or a running power meter) — don't
  treat as a safe MVP dependency without checking against a data-richer account first.
- Nutrition endpoints (`get_nutrition_daily_food_log`, `get_nutrition_daily_meals`) work but returned
  empty — they're gated on the user separately maintaining a Garmin Connect food log, which most
  runners don't do. Nutrition-aware coaching is a bigger adoption bet than it first looked, not a
  free win on top of existing data.

**Guardrail check:** every call in this spike was read-only. No `push_workout_to_device`,
`upload_*_workout`, `schedule_workout`, `set_*`, `create_*`, or `delete_*` methods were invoked.



## Follow-up spike: does `get_activity(activity_id)` carry an explicit `workoutId`?

The earlier check only looked at **top-level** keys on the *most recent* activity (which may not
have been started from a scheduled workout at all). That's not a fair test of the proposed
"query `get_activity`, look for an explicit `workoutId` key" approach. Redo it properly:

1. Pick an activity we already **know** was matched to a scheduled workout (2026-09-24, "Speed
   Repeats", found via the date cross-reference above).
2. Call `get_activity()` **and** `get_activity_details()` for it.
3. Recursively search the *entire* JSON tree (not just top-level) for any key containing
   `workout` — nested under `metadataDTO`, `summaryDTO`, etc. would otherwise be missed.



In [48]:
def find_keys_containing(obj, needle, path=""):
    """Recursively yield (path, value) for any dict key containing `needle` (case-insensitive)."""
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}" if path else k
            if needle.lower() in k.lower():
                hits.append((new_path, v))
            hits.extend(find_keys_containing(v, needle, new_path))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hits.extend(find_keys_containing(v, needle, f"{path}[{i}]"))
    return hits


known_matched_activity_id = acts_by_date["2026-09-24"]["activityId"]
print(f"Inspecting activityId={known_matched_activity_id} ('{acts_by_date['2026-09-24']['activityName']}')\n")

activity_json = client.get_activity(known_matched_activity_id)
details_json = client.get_activity_details(known_matched_activity_id)

for label, payload in [("get_activity", activity_json), ("get_activity_details", details_json)]:
    hits = find_keys_containing(payload, "workout")
    print(f"{label}: {len(hits)} 'workout'-matching key(s)")
    for path, value in hits:
        print(f"  {path} = {value!r}")
    print()


Inspecting activityId=24480686682 ('Lodi - Speed Repeats')

get_activity: 4 'workout'-matching key(s)
  metadataDTO.associatedWorkoutId = 1700418354
  metadataDTO.specializedWorkoutCategories = ['ATP']
  summaryDTO.directWorkoutFeel = 25
  summaryDTO.directWorkoutRpe = 70

get_activity_details: 0 'workout'-matching key(s)



In [49]:
# Does the lightweight list endpoint (get_activities / get_activities_by_date) already carry
# metadataDTO.associatedWorkoutId, or is the per-activity get_activity() call actually necessary?
list_item = acts_by_date["2026-09-24"]
list_hits = find_keys_containing(list_item, "workout")
print(f"get_activities_by_date list item: {len(list_hits)} 'workout'-matching key(s) -> {list_hits}")

# Cross-check: does the associatedWorkoutId equal the scheduled calendar item's workoutId?
scheduled_workout_id = next(i["workoutId"] for i in past_items if i["date"] == "2026-09-24")
associated_workout_id = activity_json["metadataDTO"]["associatedWorkoutId"]
print(f"\nscheduled calendar workoutId = {scheduled_workout_id}")
print(f"activity metadataDTO.associatedWorkoutId = {associated_workout_id}")
print(f"Exact match: {scheduled_workout_id == associated_workout_id}")

# And confirm it's genuinely absent (not just unset) on a freeform / non-workout activity.
freeform_hits = find_keys_containing(client.get_activity(recent[0]["activityId"]), "associatedWorkout")
print(f"\nrecent[0] ('{recent[0].get('activityName')}') associatedWorkoutId hits: {freeform_hits}")


get_activities_by_date list item: 1 'workout'-matching key(s) -> [('workoutId', 1700418354)]

scheduled calendar workoutId = 1700418354
activity metadataDTO.associatedWorkoutId = 1700418354
Exact match: True

recent[0] ('Lodi Walking') associatedWorkoutId hits: [('metadataDTO.associatedWorkoutId', None)]


### Correction to the earlier "no linkage field" finding

The earlier claim that activities carry no workout/plan linkage was a **false negative** — it only
checked top-level keys on a non-workout activity. Testing against an activity known to be
workout-linked shows the linkage is real and, better still, already present on the cheap **list**
call:

- `get_activities_by_date(...)` list items include `workoutId` and `isAtpActivity` directly — no
  per-activity call needed.
- `workoutId` on the activity exactly matches `workoutId` on the scheduled calendar entry
  (`1700418354` in both, for the 2026-09-24 "Speed Repeats" session).
- `get_activity(id)` also carries it (nested at `metadataDTO.associatedWorkoutId`), but is redundant
  with the list field for this purpose.
- Freeform activities (not started from a scheduled workout) have `workoutId: None` /
  `isAtpActivity: False` — a clean, explicit "not planned" signal.

**Revised Feature 2 matching plan** (supersedes the name-substring heuristic proposed earlier):
1. One `get_activities_by_date(start, end, activitytype=...)` call per sync window.
2. Join directly on `workoutId == scheduled_calendar_item["workoutId"]` — exact match, no ambiguity.
3. Fallback to date-window + type + name-substring matching only for the (rarer) activities where
   `workoutId is None` — i.e. genuinely freeform training, not device-selected workouts.

This makes the rule-based MVP for Feature 2 close to a straight join rather than a fuzzy-matching
problem, and demotes the LLM-assisted ambiguous-case path even further than previously revised.



## Spike: workouts as a first-class Bronze entity

`architecture_guidelines.md` section 4 only names two Bronze sources to investigate — **activities**
(confirmed FIT) and **wellness/monitoring** (confirmed FIT above, via `download_health_snapshot`).
Workouts/scheduled-plan data isn't called out as an entity to sync at all, even though Feature 2
(plan adherence) and Feature 3 (plan push) both depend on it being ingested and idempotently tracked
just like activities.

Check:
1. `download_workout(workout_id)` — is the workout definition available as binary FIT (like
   activities), or is `get_workout_by_id` (JSON, already confirmed) the only route?
2. Re-confirm the near-future planned side is already covered by `get_scheduled_workouts` /
   `get_next_scheduled_workout` (from the earlier spike) — so "past and near-future planned workouts"
   both have a known read path before we decide how to store them.



In [51]:
known_workout_id = past_items[-1]["workoutId"]  # "Speed Repeats", 2026-09-24

try:
    workout_bytes = client.download_workout(known_workout_id)
    print(f"download_workout({known_workout_id}) -> {type(workout_bytes)}, {len(workout_bytes):,} bytes")
    print("First 200 bytes:", workout_bytes[:200])
    is_zip = workout_bytes[:2] == b"PK"
    print(f"Looks like a zip (FIT-in-zip, like activities/wellness)? {is_zip}")
    if not is_zip:
        # Might just be raw JSON bytes rather than a binary FIT payload.
        try:
            print("Parsed as JSON:", json.loads(workout_bytes)["workoutName"])
        except Exception:
            pass
except Exception as e:
    print(f"download_workout -> ERROR: {e}")

# Sanity re-check: near-future planned workouts (not yet completed) via the calendar spike above.
print(f"\nNext scheduled workout still available: {next_workout['title']} on {next_workout['date']} (workoutId={next_workout['workoutId']})")


download_workout(1700418354) -> <class 'bytes'>, 1,272 bytes
First 200 bytes: b'\x0e\x10\xdfR\xe8\x04\x00\x00.FIT\xf8Q@\x00\x01\x00\x00\x05\x00\x01\x00\x01\x02\x84\x02\x02\x84\x04\x04\x86\x03\x04\x8c\x00\x05\x00\x01\xff\xfeE\r\x97\x92eZS2@\x00\x01\x001\x02\x01\x01\x02\x00\x02\x84\x00\x00\n:@\x00\x01\x00\x1a\x0b\x08\x0e\x07\x11.\x07\n\x04\x86\t\x01\x00\x17\x01\x00\x15\x04\x86\x04\x01\x00\x0b\x01\x00\x05\x04\x8c\x06\x02\x84\xfe\x02\x84\x00Speed Repeats\x00Warm up, fast run and jog repeats, cool down.\x00\x00+\xf2\x00\x00\x00\x00+\xf2\x00\x01\x00\x00\x00\x00 \x00\x08\x00\x00@\x00\x01\x00\x91\x05\xfa\x04\x86\x01\x02\x84\x02\x02\x84'
Looks like a zip (FIT-in-zip, like activities/wellness)? False

Next scheduled workout still available: Easy Run on 2026-09-27 (workoutId=1701701439)


### Findings — workouts belong in Bronze as a third first-class entity

- `download_workout(workout_id)` returns **raw binary FIT** directly (confirmed via the `.FIT`
  magic bytes and embedded workout name/description in the payload) — not even zipped, unlike
  activity/wellness downloads. This is a stronger FIT story than either of the other two sources.
- Combined with the earlier spikes: **all three data families Strider needs are FIT-first**:
  - Activities → `download_activity` (FIT, zipped)
  - Wellness/monitoring → `download_health_snapshot` (multiple FIT files per day: `WELLNESS`,
    `SLEEP_DATA`, `METRICS`, zipped) — this resolves the "JSON vs FIT" open question left in the
    earlier "Findings & next steps" cell in favor of FIT.
  - Workouts (definitions) → `download_workout` (raw FIT) + `get_scheduled_workouts` /
    `get_next_scheduled_workout` (JSON calendar — no FIT equivalent for the *schedule*, only for the
    workout *definition* itself, which is expected since the calendar entry is metadata, not a
    device-recordable file).
- **Near-future planned workouts are already readable** the same way as past ones — the calendar
  endpoints return both, so no separate "read the plan" mechanism is needed for past vs. upcoming.
- **Gap confirmed**: `architecture_guidelines.md` section 4 (Bronze layer) only names activities and
  wellness as sources to investigate. Workouts aren't mentioned as a Bronze entity at all, despite
  Feature 2 and Feature 3 both depending on them being ingested and idempotency-tracked the same way
  as activities. This should be added explicitly rather than left implicit — see the doc update.

